In [ ]:
# Instalação de todas as dependências
!pip install scikit-learn pandas numpy optuna xgboost scipy shap matplotlib seaborn s3fs pyarrow

In [ ]:
# ======================================
# IMPORTANDO BIBLIOTECAS
# ======================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Visualização da árvore
from sklearn.tree import plot_tree

# Normalização
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

#Métricas de classificação
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

#Feature selection
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif
from sklearn.feature_selection import RFECV

#Validação
from sklearn.model_selection import learning_curve
from sklearn.model_selection import validation_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import ks_2samp

#Validação cruzada
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import optuna
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform

#SHAP
import shap

#Imputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer



# **Treino do modelo e métricas de validação**

In [ ]:
# ======================================
# CONECTANDO NA GOLD (BASE DE DADOS)
# ======================================

# VARIAVEIS
s3_path_comparativo_metas_resultados = "s3://fiap-datalake-tech-public/gold/comparativo_metas_resultados/"


#ACESSO
dados = pd.read_parquet(
    s3_path_comparativo_metas_resultados,
    engine="pyarrow",
    storage_options={"anon": True}
)

dados.head(n=5)


,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3


In [ ]:
pib_renda = pd.read_csv('pib_renda_gold.csv')

# # Filtra apenas colunas necessárias e remove duplicatas de id_municipio
pib_renda_unico = pib_renda[[
    'id_municipio',
    'pib_per_capita_R$',
    'renda_media_per_capita_R$',
    'densidade_demografica'
]].drop_duplicates(subset=['id_municipio'])

# Merge sem gerar linhas duplicadas
dados_mais_pib_renda = dados.merge(
    pib_renda_unico,
    on='id_municipio',
    how='left'
)
dados_mais_pib_renda.head()

,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,67914.967346,402.15,2.451961
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,47065.679747,743.35,17.942874
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,49413.928396,343.73,1.599843
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12680.615355,193.40,1.183025
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12604.111612,200.40,2.116989


In [ ]:
# ======================================
# IMPUTER TRANSFORMER
# ======================================

#Colunas de meta_ano e desvio_ano não possuem valores para 2023 porque foi o primeiro ano da pesquisa (sem meta para aquele ano específico porque começou naquele momento)

imputer_num = SimpleImputer(strategy='median')

# Aplica apenas nas colunas numéricas
dados_nulos = ['renda_media_per_capita_R$','pib_per_capita_R$','densidade_demografica']
dados_mais_pib_renda[dados_nulos] = imputer_num.fit_transform(dados_mais_pib_renda[dados_nulos])

dados_mais_pib_renda.head()


,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,67914.967346,402.15,2.451961
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,47065.679747,743.35,17.942874
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,49413.928396,343.73,1.599843
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12680.615355,193.40,1.183025
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12604.111612,200.40,2.116989


In [ ]:
# ======================================
# FEATURE ENGINEERING
# ======================================
#Foi feita na Gold porque na Silver só tínhamos a meta do municipio e a taxa real, na Gold calculamos o desvio entre a meta e realidade.

#Porém, como estamos trabalhando com dados de 2023, 2024 e 2025, vamos trabalhar com séries temporais. Como nossa amostragem é de apenas 3 anos, ao tentar decompor nosso dataset nos deparamos com a dificuldade de aplicar um período ideal na biblioteca de statsmodel, para aplicar 2, teríamos que ter pelo menos 4 observações, e nós extraímos a média por ano de 3 anos, ou seja, nossa amostragem era insuficiente.
#Para o ADF também tivemos problemas por conta da amostragem pequena também.
#Com isso, como nosso dataframe possui dados temporais e o algoritmo não tem a capacidade de interpretá-lo de maneira sequencial (temporal), vamos utilizar séries temporais para utilizá-los corretamente no nosso modelo, de forma que não tenha data leakege, onde o modelo poderia prever o "passado utilizando o futuro", com dados de 2025 na previsão de 2025, por exemplo.

# 1. Garantir a ordenação temporal correta por município
dados_mais_pib_renda = dados_mais_pib_renda.sort_values(
    by=["id_municipio", "ano"]
).reset_index(drop=True)

# Criar lag_1 (valor do ano anterior: t-1) e lag_2 e diff
dados_mais_pib_renda["lag_1"] = dados_mais_pib_renda.groupby("id_municipio")[
    "taxa_alfabetizacao_real"
].shift(1)

dados_mais_pib_renda['diff_1'] = dados_mais_pib_renda['taxa_alfabetizacao_real'].diff(1)

# Criar média móvel dos 2 anos anteriores (shift(1) evita vazamento do ano corrente)
dados_mais_pib_renda["rolling_mean_2"] = (
    dados_mais_pib_renda.groupby("id_municipio")["taxa_alfabetizacao_real"]
    .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
)

dados_mais_pib_renda.head()



,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica,lag_1,diff_1,rolling_mean_2
0,2023,RO,1100015,Alta Floresta D'Oeste,227,67.40,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,48803.311567,476.99,3.033819,NaN,NaN,NaN
1,2024,RO,1100015,Alta Floresta D'Oeste,275,68.00,67.08,0.92,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,48803.311567,476.99,3.033819,67.40,0.60,67.40
2,2025,RO,1100015,Alta Floresta D'Oeste,270,78.15,69.51,8.64,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,48803.311567,476.99,3.033819,68.00,10.15,67.70
3,2023,RO,1100023,Ariquemes,1222,62.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,45592.551377,689.95,21.719449,NaN,-15.63,NaN
4,2024,RO,1100023,Ariquemes,1066,65.85,65.22,0.63,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,45592.551377,689.95,21.719449,62.52,3.33,62.52


In [ ]:
# ======================================
# FEATURE EXTRACTION
# ======================================

#Para chegarmos no Status Meta (com as classes "atingiu" ou "não atingiu") utilizamos o desvio, se era >=0 o municipio atingiu, se não, não atingiu a meta.
# O feature extraction ajudou a não ter no modelo duas features que não agregariam (meta_ano e desvio_ano) e que poderiam gerar associações espúrias e aumentar a dimensionalidade.

# Criar agregação por UF para ajudar no feature engineering já que a adição de taxa_alfabetizacao_real gera data leakage
dados_mais_pib_renda['media_alfa_uf'] = dados_mais_pib_renda.groupby('sigla_uf')['taxa_alfabetizacao_real'].transform('mean')

#Filtrar 2023 já que status meta é "Sem meta definida" e gera muito resíduo na nossa base na etapa de KNN dos lags, e alé disso, quando incluído, apresentava um F1 score de 0.16 com apenas 211 ocorrências (support)
status_desejados = ["Abaixo da Meta", "Atingiu a Meta"]
dados_mais_pib_renda = dados_mais_pib_renda[dados_mais_pib_renda['status_meta'].isin(status_desejados)].copy()
dados_mais_pib_renda.head()



,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica,lag_1,diff_1,rolling_mean_2,media_alfa_uf
1,2024,RO,1100015,Alta Floresta D'Oeste,275,68.00,67.08,0.92,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,48803.311567,476.99,3.033819,67.40,0.60,67.400,71.482949
2,2025,RO,1100015,Alta Floresta D'Oeste,270,78.15,69.51,8.64,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,48803.311567,476.99,3.033819,68.00,10.15,67.700,71.482949
4,2024,RO,1100023,Ariquemes,1066,65.85,65.22,0.63,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,45592.551377,689.95,21.719449,62.52,3.33,62.520,71.482949
5,2025,RO,1100023,Ariquemes,1325,79.70,68.02,11.68,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,45592.551377,689.95,21.719449,65.85,13.85,64.185,71.482949
7,2024,RO,1100031,Cabixi,75,76.00,70.85,5.15,Atingiu a Meta,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,56219.772005,457.17,4.072298,71.05,4.95,71.050,71.482949


In [ ]:
# ============================================================================================
# DEFININDO X e y e divisão treino e teste (Temporal: 2023-2024 -> Treino | 2025 -> Teste)
# ============================================================================================

#Definição do y
y_bruto = dados_mais_pib_renda['status_meta']

#Definição do X
colunas_X = [
    'ano',
    'lag_1',
    'diff_1',
    'rolling_mean_2',
    'total_avaliados',
    'media_alfa_uf',
    'sigla_uf',
    'renda_media_per_capita_R$',
    'pib_per_capita_R$',
    'densidade_demografica',
]
X_bruto = dados_mais_pib_renda[colunas_X]

#Feature encoding do X
  # Aplicar One-Hot Encoding en sigla_uf (Estado)
X = pd.get_dummies(X_bruto, columns=['sigla_uf'], drop_first=True, dtype=int)

#Divisão Treino (2024) e Teste (2025) baseada no ano
mascara_treino = X['ano'].isin([2024])
mascara_teste = X['ano'] == 2025

X_train = X[mascara_treino].copy()
X_test = X[mascara_teste].copy()
y_train = y_bruto[mascara_treino].copy()
y_test = y_bruto[mascara_teste].copy()

# Salvar metadados dos municípios de 2025 para a tabela final de comparação
df_resultado_2025 = dados_mais_pib_renda[mascara_teste][
    ["id_municipio", "nome_municipio", "sigla_uf", "status_meta"]
].copy()

#Remoção de coluna que não deve ir pro modelo
X_train = X_train.drop(columns=['ano']).reset_index(drop=True)
X_test = X_test.drop(columns=['ano']).reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

#Imputer transformation - Lags
#flag keep_empty_features=True ativada, preenchendo-a temporariamente com um valor padrão como 0 em vez de apagar os dados de 2023 que podem ser úteis para a generalização do modelo.
colunas_para_imputar = ['lag_1', 'diff_1', 'rolling_mean_2']

knn_imputer = KNNImputer(n_neighbors=5, weights='uniform',keep_empty_features=True)

  # Guarda as colunas originais do X_train
colunas_X = X_train.columns

  # Fit e transform em TODO o X_train
X_train_imputed = knn_imputer.fit_transform(X_train)

  # Transform em TODO o X_test
X_test_imputed = knn_imputer.transform(X_test)

  # Recria os DataFrames
X_train = pd.DataFrame(X_train_imputed, columns=colunas_X, index=X_train.index)
X_test = pd.DataFrame(X_test_imputed, columns=colunas_X, index=X_test.index)

#Feature encoding do y
le_y = LabelEncoder()
  # O fit é feito APENAS no treino
y_train_encoded = le_y.fit_transform(y_train)
  # O teste é apenas transformado (usando o mapeamento aprendido no treino)
y_test_encoded = le_y.transform(y_test)

# # Escalonamento do X (Fit no treino, transform no treino e teste)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)


#Utilizamos o StandardScale (desvio padrão) porque é indicado para Regressão Logística, já que não temos tantos outliers no conjunto de dados para justificar o uso do Robust Scaling

# ======================================
# FEATURE SELECTION: SELECT KBEST
# ======================================

# Mutual Information (escolhida porque é mais versátil, permitindo qualquer feature categórica ou não. Apesar do dataset não ser grande (não possui milhares de observações), a seleção de features vai nos ajudar na interpretabilidade)
selector = SelectKBest(mutual_info_classif, k=20)

# Fit e transform no conjunto de treino escalonado
X_train_selected = selector.fit_transform(X_train, y_train_encoded)

# Apenas transform no conjunto de teste
X_test_selected = selector.transform(X_test)

# Verificar quais features foram selecionadas
selected_names = X_train.columns[selector.get_support()]
print('Features selecionadas (Mutual Info):', selected_names.tolist())

print('Treino:', X_train_selected.shape)
print('Teste:', X_test_selected.shape)


Features selecionadas (Mutual Info): ['lag_1', 'diff_1', 'rolling_mean_2', 'total_avaliados', 'media_alfa_uf', 'renda_media_per_capita_R$', 'pib_per_capita_R$', 'densidade_demografica', 'sigla_uf_AL', 'sigla_uf_AM', 'sigla_uf_BA', 'sigla_uf_CE', 'sigla_uf_MG', 'sigla_uf_MT', 'sigla_uf_PE', 'sigla_uf_PR', 'sigla_uf_RO', 'sigla_uf_RS', 'sigla_uf_SC', 'sigla_uf_SE']
Treino: (5232, 20)
Teste: (5345, 20)


Chegamos a utilizar o wrapper RFECV que encontra o ponto ótimo via validação cruzada, ele escolheu 27 features invés de 10 com selectkbest, teve uma ótima acurácia também: 0.8492, porém, demorou muito mais do que o selectkbest. Ou seja, tem um custo computacional maior.

In [ ]:
# # =============================================================================
# # TREINAMENTO COM ÁRVORE DE DECISÃO (OTIMIZADO COM RANDOMIZED SEARCH CV)
# # =============================================================================

# Definir a grade de parâmetros a serem testados
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

# Criar o modelo e aplicar a busca
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                 # Validação cruzada com 5 folds
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_encoded)

# Melhor modelo encontrado
best_tree = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

# Avaliação com o melhor modelo
pred_best = best_tree.predict(X_test_selected)
print("Nova Acurácia:", accuracy_score(y_test_encoded, pred_best))

#--------------------------
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42,class_weight='balanced')
rf.fit(X_train_selected, y_train_encoded)

pred_rf = rf.predict(X_test_selected)
print("Acurácia Random Forest:", accuracy_score(y_test_encoded, pred_rf))


#------------------------
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb.fit(X_train_selected, y_train)

pred_gb = gb.predict(X_test_selected)
print("Acurácia Gradient Boosting:", accuracy_score(y_test, pred_gb))

Melhores parâmetros: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 8, 'min_samples_split': 2}
Nova Acurácia: 0.7612722170252573
Acurácia Random Forest: 0.7472404115996258
Acurácia Gradient Boosting: 0.775865294667914
